In [1]:
"""
Script para processar múltiplas combinações de parâmetros e salvar em CSV único.
Testa diferentes métodos de limpeza, níveis wavelet e atenuações.
"""
import pandas as pd
from ecg_utils import get_lista_pacientes
from pipeline_limpeza import processar_multiplos_pacientes
import os
from datetime import datetime


# ============================================================
# 1. FILTRAR PACIENTES VÁLIDOS
# ============================================================
print("🔍 Filtrando pacientes válidos...")

path_filtro_pacientes_csv = r'D:\cox-models-sudden-death\02_Preprocessamento_filtro\resumo_dataset_ecg.csv'
df_filtro = pd.read_csv(path_filtro_pacientes_csv, sep=',')

df_lista_validos = df_filtro[
    (df_filtro['tem_X'] == 'Sim') &
    (df_filtro['tem_Y'] == 'Sim') & 
    (df_filtro['tem_Z'] == 'Sim')
]

pacientes_validos = list(df_lista_validos['paciente'].values)
pacientes_original = get_lista_pacientes(incluir_apenas=pacientes_validos)

print(f"✅ {len(pacientes_original)} pacientes válidos\n")


# ============================================================
# 2. CONFIGURAÇÃO DO PROCESSAMENTO
# ============================================================

# Configurações básicas (FIXAS para todas as combinações)
CANAL = 'x'                    # Canal ECG: 'x', 'y' ou 'z'
MINUTOS_PULAR = 60            # Pular os primeiros 60 minutos
DURACAO_MINUTOS = 10         # Processar 240 minutos = 4 horas (120 épocas de 2 min)
SAMPLING_RATE = 200           # Taxa de amostragem
DEBUG = False                 # True para ver detalhes do processamento
OUTPUT_DIR = r'D:\cox-models-sudden-death\Arquitetura\logs\metricas_calibracao'

# MODO TESTE: Descomente para processar apenas 5 pacientes
MODO_TESTE = True
NUM_PACIENTES_TESTE = 2

# Configurações para testar (VARIÁVEIS)
METODOS_LIMPEZA = ['neurokit', 'elgendi2010', 'pantompkins1985', 'hamilton2002', 'engzeemod2012']
USAR_WAVELETS = [False, True]  # Primeiro sem wavelet, depois com wavelet
WAVELET_LEVELS = [2, 3, 4]     # Níveis de decomposição
WAVELET_ATENUACAOS = [0.2, 0.3, 0.4]  # Fatores de atenuação


# ============================================================
# 3. CALCULAR TOTAL DE COMBINAÇÕES
# ============================================================

# Sem wavelet: 5 métodos × 1 = 5 combinações
# Com wavelet: 5 métodos × 3 levels × 3 atenuações = 45 combinações
# Total: 5 + 45 = 50 combinações

num_combinacoes_sem_wavelet = len(METODOS_LIMPEZA)
num_combinacoes_com_wavelet = len(METODOS_LIMPEZA) * len(WAVELET_LEVELS) * len(WAVELET_ATENUACAOS)
total_combinacoes = num_combinacoes_sem_wavelet + num_combinacoes_com_wavelet

print("\n" + "="*70)
print("📊 RESUMO DAS CONFIGURAÇÕES")
print("="*70)
print(f"   Canal: {CANAL}")
print(f"   Duração: {DURACAO_MINUTOS} minutos ({DURACAO_MINUTOS // 2} épocas de 2 min)")
print(f"   Pacientes: {len(pacientes_original)}")
if MODO_TESTE:
    print(f"   🧪 MODO TESTE ATIVADO: Apenas {NUM_PACIENTES_TESTE} pacientes")
print(f"\n   Combinações sem wavelet: {num_combinacoes_sem_wavelet}")
print(f"   Combinações com wavelet: {num_combinacoes_com_wavelet}")
print(f"   TOTAL DE COMBINAÇÕES: {total_combinacoes}")
print(f"\n   Estimativa de linhas no CSV final:")
if MODO_TESTE:
    print(f"   ~{NUM_PACIENTES_TESTE} pacientes × {DURACAO_MINUTOS // 2} épocas × {total_combinacoes} combinações")
    print(f"   = ~{NUM_PACIENTES_TESTE * (DURACAO_MINUTOS // 2) * total_combinacoes:,} linhas".replace(',', '.'))
else:
    print(f"   ~{len(pacientes_original)} pacientes × {DURACAO_MINUTOS // 2} épocas × {total_combinacoes} combinações")
    print(f"   = ~{len(pacientes_original) * (DURACAO_MINUTOS // 2) * total_combinacoes:,} linhas".replace(',', '.'))
print("="*70)

input("\n⚠️  Pressione ENTER para continuar ou CTRL+C para cancelar...")


# ============================================================
# 4. CRIAR ARQUIVO DE SAÍDA ÚNICO
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file_epocas = f"{OUTPUT_DIR}/metricas_ecg_todas_combinacoes_epocas_{timestamp}.csv"
output_file_resumo = f"{OUTPUT_DIR}/metricas_ecg_todas_combinacoes_resumo_{timestamp}.csv"

# Garante que o diretório existe
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DataFrames para acumular todos os resultados
all_epocas = []
all_resumos = []


# ============================================================
# 5. LOOP DE PROCESSAMENTO
# ============================================================

combinacao_atual = 0
inicio_processamento = datetime.now()

# PRIMEIRO: Processar SEM wavelet
for metodo in METODOS_LIMPEZA:
    combinacao_atual += 1
    
    print("\n" + "="*70)
    print(f"🔄 COMBINAÇÃO {combinacao_atual}/{total_combinacoes}")
    print("="*70)
    print(f"   Método: {metodo}")
    print(f"   Wavelet: NÃO")
    print("="*70)
    
    # Define lista de pacientes
    pacientes_processar = pacientes_original[:NUM_PACIENTES_TESTE] if MODO_TESTE else pacientes_original
    
    try:
        df_epocas, df_resumo = processar_multiplos_pacientes(
            pacientes_lista=pacientes_processar,
            canal=CANAL,
            minutos_a_pular=MINUTOS_PULAR,
            duracao_em_minutos=DURACAO_MINUTOS,
            sampling_rate=SAMPLING_RATE,
            debug=DEBUG,
            metodo=metodo,
            usar_wavelet=False,
            wavelet_level=None,
            wavelet_atenuacao=None
        )
        
        if len(df_epocas) > 0:
            # Adiciona identificadores da combinação
            df_epocas['Combinacao_ID'] = combinacao_atual
            df_resumo['Combinacao_ID'] = combinacao_atual
            
            all_epocas.append(df_epocas)
            all_resumos.append(df_resumo)
            
            print(f"   ✅ {len(df_epocas)} épocas processadas")
        else:
            print(f"   ⚠️  Nenhuma época processada")
            
    except Exception as e:
        print(f"   ❌ ERRO: {str(e)[:100]}")
        continue


# SEGUNDO: Processar COM wavelet
for metodo in METODOS_LIMPEZA:
    for level in WAVELET_LEVELS:
        for atenuacao in WAVELET_ATENUACAOS:
            combinacao_atual += 1
            
            print("\n" + "="*70)
            print(f"🔄 COMBINAÇÃO {combinacao_atual}/{total_combinacoes}")
            print("="*70)
            print(f"   Método: {metodo}")
            print(f"   Wavelet: SIM")
            print(f"   - Level: {level}")
            print(f"   - Atenuação: {atenuacao}")
            print("="*70)
            
            # Define lista de pacientes
            pacientes_processar = pacientes_original[:NUM_PACIENTES_TESTE] if MODO_TESTE else pacientes_original
            
            try:
                df_epocas, df_resumo = processar_multiplos_pacientes(
                    pacientes_lista=pacientes_processar,
                    canal=CANAL,
                    minutos_a_pular=MINUTOS_PULAR,
                    duracao_em_minutos=DURACAO_MINUTOS,
                    sampling_rate=SAMPLING_RATE,
                    debug=DEBUG,
                    metodo=metodo,
                    usar_wavelet=True,
                    wavelet_level=level,
                    wavelet_atenuacao=atenuacao
                )
                
                if len(df_epocas) > 0:
                    # Adiciona identificadores da combinação
                    df_epocas['Combinacao_ID'] = combinacao_atual
                    df_resumo['Combinacao_ID'] = combinacao_atual
                    
                    all_epocas.append(df_epocas)
                    all_resumos.append(df_resumo)
                    
                    print(f"   ✅ {len(df_epocas)} épocas processadas")
                else:
                    print(f"   ⚠️  Nenhuma época processada")
                    
            except Exception as e:
                print(f"   ❌ ERRO: {str(e)[:100]}")
                continue


# ============================================================
# 6. CONSOLIDAR E SALVAR RESULTADOS
# ============================================================

if len(all_epocas) > 0:
    print("\n" + "="*70)
    print("💾 CONSOLIDANDO E SALVANDO RESULTADOS...")
    print("="*70)
    
    # Concatena todos os DataFrames
    df_final_epocas = pd.concat(all_epocas, ignore_index=True)
    df_final_resumo = pd.concat(all_resumos, ignore_index=True)
    
    # Salva épocas
    df_final_epocas.to_csv(output_file_epocas, index=False, sep=';', decimal=',')
    print(f"\n✅ ÉPOCAS salvas em:")
    print(f"   {output_file_epocas}")
    print(f"   Total de linhas: {len(df_final_epocas):,}".replace(',', '.'))
    print(f"   Tamanho do arquivo: {os.path.getsize(output_file_epocas) / 1024 / 1024:.2f} MB")
    
    # Salva resumo
    df_final_resumo.to_csv(output_file_resumo, index=False, sep=';', decimal=',')
    print(f"\n✅ RESUMO salvo em:")
    print(f"   {output_file_resumo}")
    print(f"   Total de linhas: {len(df_final_resumo):,}".replace(',', '.'))
    
    # Estatísticas finais
    tempo_total = datetime.now() - inicio_processamento
    print("\n" + "="*70)
    print("📈 ESTATÍSTICAS FINAIS")
    print("="*70)
    print(f"   Tempo total de processamento: {tempo_total}")
    print(f"   Combinações processadas: {len(all_epocas)}/{total_combinacoes}")
    print(f"   Total de épocas: {len(df_final_epocas):,}".replace(',', '.'))
    print(f"   Pacientes únicos: {df_final_epocas['Paciente_ID'].nunique()}")
    print(f"   Épocas por paciente (média): {len(df_final_epocas) / df_final_epocas['Paciente_ID'].nunique():.1f}")
    print(f"   HRV_MeanNN médio geral: {df_final_epocas['HRV_MeanNN'].mean():.2f} ms")
    
    # Preview
    print("\n" + "="*70)
    print("📊 PREVIEW DOS DADOS (primeiras 3 linhas)")
    print("="*70)
    print(df_final_epocas.head(3))
    
else:
    print("\n⚠️  NENHUM DADO FOI PROCESSADO!")


print("\n" + "="*70)
print("✅ SCRIPT FINALIZADO!")
print("="*70)

🔍 Filtrando pacientes válidos...
✓ Filtrado para 874 pacientes incluídos
✅ 874 pacientes válidos


📊 RESUMO DAS CONFIGURAÇÕES
   Canal: x
   Duração: 10 minutos (5 épocas de 2 min)
   Pacientes: 874
   🧪 MODO TESTE ATIVADO: Apenas 2 pacientes

   Combinações sem wavelet: 5
   Combinações com wavelet: 45
   TOTAL DE COMBINAÇÕES: 50

   Estimativa de linhas no CSV final:
   ~2 pacientes × 5 épocas × 50 combinações
   = ~500 linhas

🔄 COMBINAÇÃO 1/50
   Método: neurokit
   Wavelet: NÃO

🚀 PROCESSANDO 2 PACIENTES
   Método de limpeza: neurokit
   Filtro Wavelet: NÃO

[1/2] Processando P0001...
✅ Sinal do paciente P0001 (canal X) carregado com sucesso.
   ✅ 5 épocas processadas

[2/2] Processando P0002...
✅ Sinal do paciente P0002 (canal X) carregado com sucesso.
   ✅ 5 épocas processadas

✅ PROCESSAMENTO CONCLUÍDO
   Total de pacientes processados: 2
   Total de épocas processadas: 10

   ✅ 10 épocas processadas

🔄 COMBINAÇÃO 2/50
   Método: elgendi2010
   Wavelet: NÃO

🚀 PROCESSANDO 2 PAC

C:\Users\danie\AppData\Local\Temp\ipykernel_21840\279883317.py:218: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final_resumo = pd.concat(all_resumos, ignore_index=True)


In [7]:
df_resumo

,HRV_MeanNN_mean,HRV_MeanNN_std,HRV_MeanNN_min,HRV_MeanNN_max,HRV_MinNN_mean,HRV_MinNN_std,HRV_MinNN_min,HRV_MinNN_max,HRV_MaxNN_mean,HRV_MaxNN_std,...,Heart_Rate_max,Num_RR_Intervals_mean,Num_RR_Intervals_std,Num_RR_Intervals_min,Num_RR_Intervals_max,Paciente_ID,metodo,wavelet,wavelet_level,wavelet_atenuacao
0,563.833456,41.678094,528.933333,633.510638,305.0,0.000000,305.0,305.0,1527.0,440.831601,...,113.435846,212.0,14.645819,188,225,P0001,elgendi2010,sim,4,0.2
1,700.905391,6.319363,693.895349,709.583333,635.0,3.535534,630.0,640.0,757.0,16.046807,...,86.468370,170.0,1.581139,168,172,P0002,elgendi2010,sim,4,0.2
2,598.865561,11.541293,582.573529,610.333333,323.0,22.803509,305.0,360.0,1196.0,161.260659,...,102.991291,198.8,3.633180,195,204,P0003,elgendi2010,sim,4,0.2
3,795.484845,15.283409,769.935484,806.891892,666.0,49.924944,610.0,725.0,1042.0,342.811902,...,77.928607,149.8,3.033150,148,155,P0004,elgendi2010,sim,4,0.2
4,793.575482,37.317091,740.031250,830.454545,504.0,104.127326,390.0,585.0,1058.0,33.652637,...,81.077657,150.2,6.906519,143,160,P0005,elgendi2010,sim,4,0.2


In [ ]:
import pandas as pd

from ecg_utils import get_lista_pacientes, load_ecg_segment  # biblioteca local


# --- PREPARAÇÃO DOS DADOS DE REFERÊNCIA ---
path_clinical_csv = r'D:\cox-models-sudden-death\01_Dataset\dados_csv_info_definitions\subject-info_formatado.csv'
df_clinical = pd.read_csv(path_clinical_csv, sep=';')

column_mapping = {
    'Patient ID': 'paciente_id', 'Average RR (ms)': 'HRV_MeanNN',
    'minimum RR (ms)': 'HRV_MinNN', 'maximum RR (ms)': 'HRV_MaxNN',
    'RR range (ms)': 'RR_Range', 'Bradycardia': 'Bradycardia',
    'SDNN (ms)': 'HRV_SDNN', 'RMSSD (ms)': 'HRV_RMSSD',
    'pNN50 (%)': 'HRV_pNN50', 'QRS duration (ms)': 'QRS_Duration_Mean',
    'QT interval (ms)': 'QT_Interval_Mean'
}
df_references = df_clinical[column_mapping.keys()].copy()
df_references.rename(columns=column_mapping, inplace=True)z
df_references.set_index('paciente_id', inplace=True)

#-------- filtros dos pacientes ----------------

path_filtro_pacientes_csv = r'D:\cox-models-sudden-death\02_Preprocessamento_filtro\resumo_dataset_ecg.csv'
df_filtro = pd.read_csv(path_filtro_pacientes_csv, sep=',')

df_lista_validos = df_filtro[(df_filtro['tem_X'] == 'Sim') &
                             (df_filtro['tem_Y'] == 'Sim') & 
                             (df_filtro['tem_Z'] == 'Sim')]
    
pacientes_validos = list(df_lista_validos['paciente'].values)

pacientes_para_processar = get_lista_pacientes(incluir_apenas=pacientes_validos)


sinal = load_ecg_segment('P0002', canal='x', minutos_a_pular=60, duracao_em_minutos=10, sampling_rate=200)

#ecg_limpo = nk.ecg_clean(sinal, sampling_rate=200, method='elgendi2010')
dados = ECGMetricCalculator(sinal, sampling_rate=200) # ,debug=False
teste = dados.calculate_metrics_for_epochs()

✓ Filtrado para 874 pacientes incluídos
✅ Sinal do paciente P0002 (canal X) carregado com sucesso.
📊 Sinal recebido: 120000 amostras
✅ Sinal limpo: 120000 amostras
   Duração da época: 120s (24000 amostras)
   Número de épocas possíveis: 5
📦 Épocas criadas: 5 épocas
   Nomes das épocas: [np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5')]

🔍 INICIANDO PROCESSAMENTO DAS ÉPOCAS

📌 Processando época '1'...
   Tamanho do sinal da época: 24000 amostras
   ✓ Picos R encontrados: 173
   ✓ Método 1 (nk.hrv_time) processado
   ⚠️  Método 2 falhou, usando Método 1 como fallback
   ✓ Métricas morfológicas processadas
   ✅ Época processada com sucesso!

📌 Processando época '2'...
   Tamanho do sinal da época: 24000 amostras
   ✓ Picos R encontrados: 171
   ✓ Método 1 (nk.hrv_time) processado
   ⚠️  Método 2 falhou, usando Método 1 como fallback
   ✓ Métricas morfológicas processadas
   ✅ Época processada com sucesso!

📌 Processando época '3'...
   Tamanho do sinal da época: 24000

In [9]:
teste

,Epoca,HRV_MeanNN_v1,HRV_MinNN_v1,HRV_MaxNN_v1,RR_Range_v1,HRV_SDNN_v1,HRV_RMSSD_v1,HRV_pNN50_v1,HRV_MeanNN_v2,HRV_MinNN_v2,HRV_MaxNN_v2,RR_Range_v2,Bradycardia_v2,HRV_SDNN_v2,HRV_RMSSD_v2,HRV_pNN50_v2,QRS_Duration_Mean,QT_Interval_Mean
0,1,693.895349,635.0,755.0,120.0,28.075724,11.584927,0.581395,693.895349,635.0,755.0,120.0,0,28.075724,11.584927,0.581395,78.352601,234.404829
1,2,701.058824,630.0,780.0,150.0,27.884951,10.748607,0.000000,701.058824,630.0,780.0,150.0,0,27.884951,10.748607,0.000000,84.678363,229.471171
2,3,695.906433,640.0,735.0,95.0,19.210062,6.870654,0.000000,695.906433,640.0,735.0,95.0,0,19.210062,6.870654,0.000000,78.837209,227.830072
3,4,709.613095,635.0,760.0,125.0,25.914397,8.538395,0.000000,709.613095,635.0,760.0,125.0,0,25.914397,8.538395,0.000000,79.047619,229.542326
4,5,704.112426,635.0,755.0,120.0,25.683150,8.720187,0.000000,704.112426,635.0,755.0,120.0,0,25.683150,8.720187,0.000000,78.852941,223.590157


In [92]:
teste

""


In [12]:
dados

In [ ]:
import pandas as pd

link = r'D:\cox-models-sudden-death\01_Dataset\dados_csv_info_definitions\subject-info_formatado.csv'


df = pd.read_csv(link, sep=';')


df[['Patient ID',
    'minimum RR (ms)',
    'Average RR (ms)',
    'maximum RR (ms)',
    'RR range (ms)',
    'Average RR (ms)',
    'Bradycardia',
    'SDNN (ms)',
    'SDANN (ms)',
    'RMSSD (ms)',
    'pNN50 (%)',
    'QRS duration (ms)',
    'QT interval (ms)'
]]

In [91]:
df

,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,SCD_4years SinusRhythm,HF_4years SinusRhythm,Age,Gender (male=1),Weight (kg),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,0,0,58,1,83,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,0,0,58,1,74,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,0,0,69,1,83,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,0,0,56,0,84,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,0,0,70,1,97,...,0,1,1,0,1,0,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
987,P1064,1393,1393,NaN,0,0,0,63,0,67,...,1,1,1,0,1,0,0,0,0,0
988,P1065,1393,1393,NaN,0,0,0,80,1,79,...,0,1,0,1,1,0,0,0,1,0
989,P1066,1387,1387,NaN,0,0,0,72,0,81,...,0,0,0,0,1,0,0,0,1,0
990,P1073,1365,1365,NaN,0,0,0,70,0,63,...,0,1,1,0,1,0,0,0,1,0


In [87]:
# Métrica Desejada (do CSV Clínico)	Coluna Correspondente no NeuroKit2	Função NeuroKit2 a ser Usada
# Average RR (ms)	HRV_MeanNN	nk.hrv_time()
# minimum RR (ms)	HRV_MinNN	nk.hrv_time()
# maximum RR (ms)	HRV_MaxNN	nk.hrv_time()
# RR range (ms)	(Cálculo: HRV_MaxNN - HRV_MinNN)	nk.hrv_time()
# Bradycardia	(Cálculo: ECG_Rate_Mean < 60)	nk.hrv_time() (usa a coluna ECG_Rate_Mean)
# SDNN (ms)	HRV_SDNN	nk.hrv_time()
# RMSSD (ms)	HRV_RMSSD	nk.hrv_time()
# pNN50 (%)	HRV_pNN50	nk.hrv_time()
# QRS duration (ms)	ECG_Rate_Mean_QRS_Duration	nk.ecg_intervalrelated()
# QT interval (ms)	ECG_Rate_Mean_QT_Interval	nk.ecg_intervalrelated()




# HRV_MeanNN
# HRV_MinNN
# HRV_MaxNN
# RR range (ms)	(Cálculo: HRV_MaxNN - HRV_MinNN)
# Bradycardia
# HRV_SDNN
# HRV_pNN50

In [65]:
import neurokit2 as nk

# Download data
data = nk.data("bio_resting_5min_100hz")

# Process the data
df, info = nk.ecg_process(data["ECG"], sampling_rate=100)

# Single dataframe is passed
teste = nk.ecg_intervalrelated(df, sampling_rate=100)
 




In [ ]:
hrv_metrics2 = nk.ecg_intervalrelated(df, sampling_rate=100)

# peaks, info = nk.ecg_peaks(df, sampling_rate=100)

# hrv_metrics2 = nk.hrv(peaks, sampling_rate=100, show=False)

# ==========================================================
# ✅ NOVO: Extração de todas as métricas de HRV desejadas
# ==========================================================
hrv_mean_nnV2 = hrv_metrics2['HRV_MeanNN'].values[0]
hrv_min_nnV2 = hrv_metrics2['HRV_MinNN'].values[0]
hrv_max_nnV2 = hrv_metrics2['HRV_MaxNN'].values[0]
rr_rangeV2 = hrv_max_nnV2 - hrv_min_nnV2  # Cálculo derivado
bradycardiaV2 = int(hrv_metrics2['ECG_Rate_Mean'].values[0] < 60) # Cálculo derivado
hrv_sdnnV2 = hrv_metrics2['HRV_SDNN'].values[0]
hrv_rmssdV2 = hrv_metrics2['HRV_RMSSD'].values[0]
hrv_pnn50V2 = hrv_metrics2['HRV_pNN50'].values[0]


# # ===== Detecção de Bradicardia (frequência < 60 bpm) =====
# avg_heart_rate = 60000 / avg_rr
# bradycardia = int(avg_heart_rate < 60)

In [85]:
hrv_metrics2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 92 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   ECG_Rate_Mean                 1 non-null      object
 1   HRV_MeanNN                    1 non-null      object
 2   HRV_SDNN                      1 non-null      object
 3   HRV_SDANN1                    1 non-null      object
 4   HRV_SDNNI1                    1 non-null      object
 5   HRV_SDANN2                    1 non-null      object
 6   HRV_SDNNI2                    1 non-null      object
 7   HRV_SDANN5                    1 non-null      object
 8   HRV_SDNNI5                    1 non-null      object
 9   HRV_RMSSD                     1 non-null      object
 10  HRV_SDSD                      1 non-null      object
 11  HRV_CVNN                      1 non-null      object
 12  HRV_CVSD                      1 non-null      object
 13  HRV_MedianNN            

In [ ]:
[
    'HRV_MeanNN',
    'HRV_RMSSD',
    'HRV_SDNN',
    'HRV_SDANN',
    'HRV_pNN50',
    'ECG_Rate_Mean_QRS_Duration',
    'ECG_Rate_Mean_QT_Interval'
]


[
    'minimum RR (ms) ',
    'Average RR (ms)',
    'maximum RR (ms)',
    'RR range (ms)',
    'Average RR (ms)',
    'Bradycardia',
    'SDNN (ms)',
    'SDANN (ms)',
    'RMSSD (ms)',
    'pNN50 (%)'
    'QRS duration (ms)',
    'QT interval (ms)'
]



In [ ]:
eletrocardiogramas_holter = [
    'Hig-resolution ECG available',
    'ECG rhythm ',


    'QRS duration (ms)',
    'QT interval (ms)',

]


holter = [
    'Holter available',
    'Holter onset (hh:mm:ss)',
    'Holter  rhythm ',
    'minimum RR (ms) ',
    'Average RR (ms)',
    'maximum RR (ms)',
    'RR range (ms)',
    'Number of ventricular premature beats in 24h',
    'Extrasystole couplets ',
    'Ventricular Extrasystole',
    'Non-sustained ventricular tachycardia (CH>10)',
    'Longest RR pause (ms)',
    'Bradycardia',
    'SDNN (ms)',
    'SDANN (ms)',
    'RMSSD (ms)',
    'pNN50 (%)'
]

Index(['Patient ID', 'Follow-up period from enrollment (days)', 'days_4years',
       'Exit of the study', 'Cause of death', 'SCD_4years SinusRhythm',
       'HF_4years SinusRhythm', 'Age', 'Gender (male=1)', 'Weight (kg)',
       ...
       'Angiotensin-II receptor blocker (yes=1)',
       'Anticoagulants/antitrombotics  (yes=1)', 'Betablockers (yes=1)',
       'Digoxin (yes=1)', 'Loop diuretics (yes=1)', 'Spironolactone (yes=1)',
       'Statins (yes=1)', 'Hidralazina (yes=1)', 'ACE inhibitor (yes=1)',
       'Nitrovasodilator (yes=1)'],
      dtype='object', length=105)

In [ ]:
df['HRV_MeanNN']


'HRV_MeanNN': hrv['HRV_MeanNN'].values[0],
'HRV_RMSSD': hrv['HRV_RMSSD'].values[0],
'QRS_Duration_Mean': qrs['ECG_Rate_Mean_QRS_Duration'].values[0],
'QT_Interval_Mean': qrs['ECG_Rate_Mean_QT_Interval'].values[0] 

KeyError: 'HRV_MeanNN'